# DataFrames 3: Arrow schemas and record batches

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/dataframes/dataframes-03-arrow.ipynb)

Use Arrow to preserve an explicit schema and move batches between processing tools.

*Last reviewed: November 2026, against PyArrow 26.*

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q pyarrow~=26.0

## Setup and sample data

Three orders include a null amount. The schema explicitly sets the width of the key and amount.

In [ ]:
import pyarrow as pa
import pyarrow.compute as pc

schema = pa.schema([("id", pa.int64()), ("cents", pa.int64())])
orders = pa.Table.from_pylist(
    [{"id": 1, "cents": 1200}, {"id": 2, "cents": None}, {"id": 3, "cents": 800}],
    schema=schema,
)
print(orders.schema)

## Work with typed columns

Arrow represents nullable values separately from the value buffers. Compute kernels can process a column with its type and null information intact.

**Where it matters**

Keep nullable integer amounts across an interchange boundary.

**Watch out for**

Check types after a conversion. Casts, layout changes, and some consumer formats can allocate copies.

In [ ]:
amounts = orders["cents"]
print("type:", amounts.type, "nulls:", amounts.null_count)
print("known total:", pc.sum(amounts).as_py())

**What to notice:** The column stays int64 with one null. Summing known amounts gives 2000 cents.

## Pass data in record batches

A record batch groups equal-length arrays with one schema. A stream of batches can deliver rows without requiring the consumer to materialize the whole dataset.

**Where it matters**

Process one batch at a time when the full result would be too large.

**Watch out for**

An individual batch still needs memory, and converting all batches into Python lists removes the memory benefit.

In [ ]:
reader = pa.RecordBatchReader.from_batches(orders.schema, orders.to_batches(max_chunksize=2))
for batch in reader:
    print("rows:", batch.num_rows, "ids:", batch.column(0).to_pylist())

**What to notice:** The consumer sees IDs 1 and 2 in the first batch and ID 3 in the second.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Filter orders to known amounts using a validity mask. Expect IDs 1 and 3.

In [ ]:
# Try your solution here.

In [ ]:
print(orders.filter(pc.is_valid(orders["cents"]))["id"].to_pylist())

## Key takeaways

- Keep nullable integer amounts across an interchange boundary.
- Process one batch at a time when the full result would be too large.

**Next:** [DataFrames 4: Choosing an engine](dataframes-04-choosing-an-engine.ipynb)